# Preprocessing, EDA, and WOE/IV Analysis
This notebook covers steps 11 to 23 of the credit risk project:
11. Exploratory Data Analysis
12. Default Rate Analysis
13. Numerical Variable Analysis
14. Train / Validation / Test Split
15-20. WOE, IV, and Binning Concepts
21. Implement WOE / IV
22. Variable Screening Using IV
23. Multicollinearity

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

# Add src to path to import woe_iv
sys.path.append(os.path.abspath('../src'))
from woe_iv import calculate_woe_iv, transform_to_woe

import warnings
warnings.filterwarnings('ignore')

## Load Data and Target Variable Definition
Y = 1 -> Bad / Default
Y = 0 -> Good

In [ ]:
# Assuming the dataset is space-separated or comma-separated. 
# German Credit Data (Statlog) has no header by default.
# We will define the columns based on standard German credit data descriptions.

columns = [
    'checking_account', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_account', 'employment', 'installment_rate', 'personal_status_sex',
    'other_debtors', 'residence_since', 'property', 'age', 'other_installment_plans',
    'housing', 'existing_credits', 'job', 'num_dependents', 'telephone', 'foreign_worker', 'target'
]

# Adjust path if necessary
df = pd.read_csv('../data/raw/german.data', sep=' ', header=None, names=columns)

# Target in german dataset: 1=Good, 2=Bad. We map to 0=Good, 1=Bad as per instructions
df['target'] = df['target'].map({1: 0, 2: 1})

# Verify target encoding
print(df["target"].value_counts())

default_rate = df["target"].mean()
print("Default Rate:", default_rate)

## 11. Exploratory Data Analysis

In [ ]:
# Numerical variables summary
numerical_cols = ['duration', 'credit_amount', 'installment_rate', 'residence_since', 'age', 'existing_credits', 'num_dependents']
display(df[numerical_cols].describe())

# Categorical variables summary
categorical_cols = [col for col in df.columns if col not in numerical_cols and col != 'target']
for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

## 12. Default Rate Analysis

In [ ]:
# Example: default rate by housing
display(pd.crosstab(df["housing"], df["target"], normalize="index"))

plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="housing", y="target", ci=None)
plt.title('Default Rate by Housing')
plt.show()

## 13. Numerical Variable Analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="target", y="credit_amount")
plt.title('Credit Amount Distribution by Target')
plt.show()

# You can do this for other numerical variables too: Age, duration, etc.
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(['age', 'duration', 'installment_rate', 'existing_credits', 'num_dependents']):
    sns.boxplot(data=df, x="target", y=col, ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

## 14. Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# First split: Train (70%) and Temp (30%)
train, temp = train_test_split(
    df,
    test_size=0.30,
    stratify=df["target"],
    random_state=42
)

# Second split: Validation (15%) and Test (15%)
validation, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp["target"],
    random_state=42
)

print(f"Train shape: {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"Test shape: {test.shape}")

## 18 & 19. Binning Variables (Train Data)

In [ ]:
# Age Binning
train['age_binned'] = pd.cut(train['age'], bins=[17, 25, 35, 45, 55, 120], labels=['18-25', '26-35', '36-45', '46-55', '56+'])

# Duration Binning using qcut
train['duration_binned'] = pd.qcut(train['duration'], q=5, duplicates="drop").astype(str)

# Combine rare categories (example for housing)
# A151: rent, A152: own, A153: for free
# This is already quite grouped, but here's how you might combine:
train['housing_grouped'] = train['housing'].replace({'A153': 'A151_A153'})

## 21. Implement & Apply WOE / IV

In [ ]:
# Apply our WOE / IV function to the binned age
age_woe_rules = calculate_woe_iv(train, 'age_binned', 'target')
display(age_woe_rules)

# Transform training data
train = transform_to_woe(train, 'age_binned', age_woe_rules)
display(train[['age', 'age_binned', 'age_binned_WOE']].head())

## 22. Variable Screening Using IV

In [ ]:
# Calculate IV for categorical and binned variables
iv_results = []
candidate_features = categorical_cols + ['age_binned', 'duration_binned']

# Ensure binned variables exist in train
for f in candidate_features:
    try:
        rules = calculate_woe_iv(train, f, 'target')
        total_iv = rules['IV'].sum()
        decision = 'Keep' if total_iv >= 0.02 else 'Remove'
        if total_iv > 0.5:
            decision = 'Investigate (Suspiciously High)'
        iv_results.append({'Feature': f, 'IV': total_iv, 'Decision': decision})
    except Exception as e:
        pass

iv_df = pd.DataFrame(iv_results).sort_values(by='IV', ascending=False)
display(iv_df)

## 23. Multicollinearity (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Select some numerical variables to check VIF
features_vif = train[['duration', 'credit_amount', 'installment_rate', 'age']].dropna()
features_vif['intercept'] = 1 # Required for VIF

vif_data = pd.DataFrame()
vif_data["Feature"] = features_vif.columns
vif_data["VIF"] = [variance_inflation_factor(features_vif.values, i) for i in range(len(features_vif.columns))]

display(vif_data[vif_data['Feature'] != 'intercept'])

# Also check correlation
plt.figure(figsize=(6, 4))
sns.heatmap(train[['duration', 'credit_amount', 'installment_rate', 'age']].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()